# BraTS 2021 nnU-Net (DynUNet) Baseline — Full Dataset Runner on Kaggle

Notebook chuẩn hóa chạy **nnU-Net (DynUNet) Baseline trên TOÀN BỘ tập dữ liệu BraTS 2021 (1,251 ca: ~1,000 Train / ~125 Val / ~126 Held-Out Test)**.

### Đặc tả cấu hình nnU-Net Baseline:
- **Kiến trúc mạng**: **`DynUNet` (MONAI)** với chuẩn hóa **`InstanceNorm3d`** (không phụ thuộc batch size, triệt tiêu hoàn toàn bất ổn định phương sai) và kích hoạt **`LeakyReLU(0.01)`**.
- **Độ chính xác**: Thuần **`FP32` (`--no-amp`)** cho sự ổn định số học tối đa trên GPU NVIDIA T4.
- **Phần cứng**: **`GPU T4 x2`** (`torch.nn.DataParallel` tự động nhận diện cả 2 GPU) + 4 CPU workers (`persistent_workers=True`).
- **Lịch trình huấn luyện**: **`65` epochs**, 5 mốc Validation (`val_interval=13`: epochs 13, 26, 39, 52, 65). Tự động lưu checkpoint tốt nhất theo Validation Mean Dice.
- **Dữ liệu**: Nhấn **`+ Add Input`** -> chọn **`brats-2021-task1`** (`dschettler8845/brats-2021-task1`). Toàn bộ 1,251 ca trong `BraTS2021_Training_Data.tar` tự động giải nén vào `/kaggle/tmp` (ổ cứng SSD tạm), tiêu tốn **0 MB** dung lượng `/kaggle/working`.
- **Đánh giá Held-Out Test**: Tự động đánh giá độc lập trên toàn bộ ~126 ca tập Test (10%) chưa từng xuất hiện trong quá trình train/val.

In [ ]:
# 1. Kiểm tra tài nguyên GPU T4 x2, CPU RAM và phân vùng ổ đĩa tạm /kaggle/tmp
!nvidia-smi
!df -h /kaggle/working /tmp /dev/shm
!find /kaggle/input -maxdepth 3

In [ ]:
# 2. Cài đặt các thư viện y tế chuyên sâu (MONAI, NiBabel) & Clone repo chính thức
!pip install -q monai nibabel scipy matplotlib tqdm
!rm -rf /kaggle/tmp/repo && mkdir -p /kaggle/tmp
# Clone từ branch baselineTrain của repository nhóm (fallback sang personal repo nếu cần)
!git clone -b baselineTrain https://github.com/nam21-12-06/test.git /kaggle/tmp/repo || git clone -b baselineTrain https://github.com/chitruonghf711/brats2021-3d-unet-study.git /kaggle/tmp/repo

In [ ]:
# 3. Huấn luyện nnU-Net Baseline trên TOÀN BỘ 1,251 ca BraTS 2021
#    - DynUNet với InstanceNorm3d + LeakyReLU
#    - Tự động chạy DataParallel 2x T4 GPU, batch size 4
#    - Thuần FP32 (--no-amp) bảo đảm gradient ổn định
#    - Sau khi hoàn thành, tự động đánh giá Held-Out Test trên toàn bộ ~126 ca test
!python -u /kaggle/tmp/repo/scripts/run_nnunet_baseline.py \
    --epochs 65 \
    --val-interval 13 \
    --roi-size 96 \
    --batch-size 4 \
    --sw-batch-size 4 \
    --lr 1e-3 \
    --optimizer adamw \
    --loss-type dice_bce \
    --overlap 0.5 \
    --cache-rate 0.0 \
    --num-workers 4 \
    --no-amp \
    --keep-latest \
    --max-hours 11.5 \
    --output-dir /kaggle/working/outputs/nnunet \
    --checkpoint-dir /kaggle/working/checkpoints/nnunet

In [ ]:
# 4. Đọc và trực quan hóa toàn bộ kết quả thực nghiệm Held-Out Test & Biểu đồ huấn luyện
import json
from IPython.display import Image, display

report_path = "/kaggle/working/outputs/nnunet/nnunet_metrics_report.json"
try:
    with open(report_path, "r", encoding="utf-8") as f:
        rep = json.load(f)

    print(f"=== KẾT QUẢ THỰC NGHIỆM nnU-Net (DynUNet) TRÊN TOÀN BỘ BraTS 2021 ===")
    print(f"Tổng số ca bệnh nhân: {rep['dataset_split']['total_cases']}")
    print(f"Phân chia tập: Train={rep['dataset_split']['train_cases']}, Val={rep['dataset_split']['val_cases']}, Test={rep['dataset_split']['test_cases']}")
    print(f"Epoch Val tốt nhất: {rep['best_val_epoch']} | Best Val Mean Dice: {rep['best_val_dice_mean']:.4f}")
    print("\n--- TỔNG HỢP KẾT QUẢ TẬP HELD-OUT TEST (MEAN) ---")
    print(json.dumps(rep["test_summary_mean"], indent=2))
    print("\n--- PHÂN PHỐI THỐNG KÊ TẬP TEST (MEAN / STD / MEDIAN) ---")
    print(json.dumps(rep["test_distribution_stats"], indent=2))

    display(Image(filename="/kaggle/working/outputs/nnunet/nnunet_learning_curves.png"))
    display(Image(filename="/kaggle/working/outputs/nnunet/nnunet_test_predictions.png"))
except Exception as e:
    print("Lỗi hiển thị kết quả:", e)